# Chapter 5: Multi-Token Prediction and FP8 Quantization

# Cell 5.1: Import Required Libraries

In [1]:
# Imports the built-in math library.
# Used for mathematical operations such as square roots,
# logarithms, exponentials, etc.
import math

# Imports Optional type from Python's typing module.
# Used in function definitions to indicate that a parameter
# can either have a value or be None.
from typing import Optional

# Imports the main PyTorch library.
# Provides tensor operations and GPU acceleration.
import torch

# Imports PyTorch's neural network module.
# Contains layers such as Linear, Embedding, LayerNorm,
# Module, ModuleList, etc.
import torch.nn as nn

# Imports PyTorch neural network functions.
# Contains activation functions and utility functions such as:
# softmax(), gelu(), relu(), cross_entropy(), etc.
import torch.nn.functional as F

# Part 1: Multi-Token Prediction (MTP)

# Implementing a Causal Multi-Token Prediction Module from Scratch

We will build the entire MTP system in stages:

1. **RMSNorm**: The normalization layer used throughout the DeepSeek architecture
2. **The MTP Module**: The core component that processes one step in the causal chain
3. **The Main Model**: A wrapper class that integrates the main Transformer trunk and the chain of MTP modules
4. **The Forward Pass & Loss**: The complete logic for sequential prediction and the combined loss calculation

# Listing 5.1: RMSNorm

In [2]:
class RMSNorm(nn.Module):
    """
    Implements Root Mean Square Layer Normalization.

    RMSNorm scales activations based on their root mean square
    value without subtracting the mean, making it simpler and
    computationally cheaper than LayerNorm.
    """

    # Constructor
    def __init__(self, d_model: int, eps: float = 1e-6):

        # Initialize parent nn.Module class
        super().__init__()

        # Small constant added for numerical stability
        # Prevents division by zero during normalization
        self.eps = eps

        # Learnable scaling parameter γ (gamma)
        # Shape: [d_model]
        # Initially all values are 1
        self.weight = nn.Parameter(torch.ones(d_model))

    # Forward pass
    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # Compute RMS normalization
        #
        # Step 1: x.pow(2)
        #   Squares each element
        #
        # Step 2: mean(-1, keepdim=True)
        #   Computes mean across embedding dimension
        #
        # Step 3: + self.eps
        #   Avoids division by zero
        #
        # Step 4: torch.rsqrt(...)
        #   Computes 1/sqrt(value)
        #
        # Step 5: x * rsqrt(...)
        #   Normalizes the tensor
        norm_x = x * torch.rsqrt(
            x.pow(2).mean(-1, keepdim=True) + self.eps
        )

        # Apply learnable scaling weight γ
        # Shape remains same as input
        return self.weight * norm_x

# Listing 5.2: The Causal DeepSeek Multi-Token Prediction (MTP) Module

In [3]:
class DeepSeekMTPModule(nn.Module):

    # Constructor
    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        dropout: float = 0.0
    ):

        # Initialize parent PyTorch module
        super().__init__()

        # Store embedding dimension
        self.d_model = d_model

        # ----------------------------------------------------
        # Projection Layer
        # ----------------------------------------------------
        # Input:
        #   [hidden_state ; future_token_embedding]
        #
        # Shape:
        #   (2*d_model) → d_model
        #
        # Used to compress concatenated information back
        # to the original hidden dimension.
        self.projection_matrix = nn.Linear(
            2 * d_model,
            d_model,
            bias=False
        )

        # ----------------------------------------------------
        # Transformer Refinement Block
        # ----------------------------------------------------
        # A standard Transformer Encoder layer.
        #
        # Used to mix information and generate a refined
        # hidden representation.
        self.transformer_block = nn.TransformerEncoderLayer(

            # Embedding dimension
            d_model=d_model,

            # Number of attention heads
            nhead=nhead,

            # Feed Forward Network size
            dim_feedforward=dim_feedforward,

            # Dropout probability
            dropout=dropout,

            # Activation function
            activation='gelu',

            # Input shape:
            # (batch, seq_len, hidden_dim)
            batch_first=True,

            # Pre-Norm Transformer
            norm_first=True
        )

        # ----------------------------------------------------
        # RMSNorm for previous hidden state
        # ----------------------------------------------------
        self.norm_hidden = RMSNorm(d_model)

        # ----------------------------------------------------
        # RMSNorm for future token embedding
        # ----------------------------------------------------
        self.norm_embed = RMSNorm(d_model)

    # ========================================================
    # Forward Pass
    # ========================================================
    def forward(
        self,
        h_prev: torch.Tensor,
        future_token_embeds: torch.Tensor
    ) -> torch.Tensor:

        # Normalize previous hidden state
        h_normed = self.norm_hidden(h_prev)

        # Normalize future token embedding
        embed_normed = self.norm_embed(
            future_token_embeds
        )

        # ----------------------------------------------------
        # Concatenate both representations
        #
        # Before:
        # h_normed      → (B,S,D)
        # embed_normed  → (B,S,D)
        #
        # After:
        # concatenated  → (B,S,2D)
        # ----------------------------------------------------
        concatenated = torch.cat(
            [h_normed, embed_normed],
            dim=-1
        )

        # Project back to D dimensions
        #
        # (B,S,2D) → (B,S,D)
        h_prime = self.projection_matrix(
            concatenated
        )

        # Refine representation using Transformer
        h_output = self.transformer_block(
            h_prime
        )

        # Return refined hidden state
        return h_output

### Listings 5.3 & 5.4: The Full MTP Model Architecture with Forward Pass

In [4]:
class DeepSeekV3WithMTP(nn.Module):

    # ========================================================
    # Constructor
    # ========================================================
    def __init__(
        self,
        vocab_size: int,          # Vocabulary size
        d_model: int,             # Embedding dimension
        num_layers: int,          # Number of Transformer layers
        nhead: int,               # Number of attention heads
        num_mtp_heads: int,       # Number of MTP prediction depths (D)
        dim_feedforward: int,     # FFN hidden dimension
        dropout: float = 0.0,     # Dropout probability
        mtp_loss_weight: float = 0.1  # λ (lambda) in DeepSeek paper
    ):

        # Initialize parent class
        super().__init__()

        # Store model hyperparameters
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.nhead = nhead
        self.num_mtp_heads = num_mtp_heads
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.mtp_loss_weight = mtp_loss_weight

        # ----------------------------------------------------
        # Shared Token Embedding Layer
        # ----------------------------------------------------
        # Converts token IDs → dense vectors
        #
        # Shape:
        # (vocab_size) → (d_model)
        self.shared_embed = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Shared Language Modeling Head
        # ----------------------------------------------------
        # Converts hidden states → vocabulary logits
        #
        # Shape:
        # d_model → vocab_size
        self.shared_lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # ----------------------------------------------------
        # Main Transformer Backbone
        # ----------------------------------------------------
        # DeepSeek calls this the Shared Transformer Trunk
        #
        # Multiple Transformer Encoder blocks are stacked
        # together.
        self.blocks = nn.ModuleList([

            nn.TransformerEncoderLayer(

                # Hidden dimension
                d_model=d_model,

                # Number of attention heads
                nhead=nhead,

                # Feed Forward size
                dim_feedforward=dim_feedforward,

                # Dropout
                dropout=dropout,

                # Activation function
                activation='gelu',

                # Input shape:
                # (batch, seq_len, hidden_dim)
                batch_first=True,

                # Pre-normalization
                norm_first=True
            )

            # Create num_layers Transformer blocks
            for _ in range(num_layers)
        ])

        # Final RMSNorm layer
        self.norm_f = RMSNorm(d_model)

        # ----------------------------------------------------
        # Weight Tying
        # ----------------------------------------------------
        # Input embedding matrix and output projection matrix
        # share the same parameters.
        #
        # Benefits:
        # - Fewer parameters
        # - Better generalization
        self.shared_lm_head.weight = (
            self.shared_embed.weight
        )

        # ----------------------------------------------------
        # Multi-Token Prediction Modules
        # ----------------------------------------------------
        # Creates D MTP modules
        #
        # Example:
        # D=3
        # MTP1 predicts token t+2
        # MTP2 predicts token t+3
        # MTP3 predicts token t+4
        self.mtp_modules = nn.ModuleList([

            DeepSeekMTPModule(
                d_model,
                nhead,
                dim_feedforward,
                dropout
            )

            for _ in range(num_mtp_heads)
        ])

    # ========================================================
    # Shared Embedding Lookup
    # ========================================================
    def get_embedding(
        self,
        input_ids: torch.Tensor
    ) -> torch.Tensor:

        """
        Converts token IDs into embeddings.
        Shared by:
        - Main Transformer
        - All MTP modules
        """

        return self.shared_embed(input_ids)

    # ========================================================
    # Shared Output Projection
    # ========================================================
    def get_output_logits(
        self,
        hidden_states: torch.Tensor
    ) -> torch.Tensor:

        """
        Converts hidden states into vocabulary logits.
        Shared by:
        - Main prediction head
        - All MTP heads
        """

        return self.shared_lm_head(hidden_states)

    # ========================================================
    # Forward Pass
    # ========================================================
    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None
    ):

        # ----------------------------------------------------
        # Input shape
        # ----------------------------------------------------
        # B = Batch Size
        # S = Sequence Length
        B, S = input_ids.shape

        # ====================================================
        # MAIN MODEL FORWARD PASS
        # ====================================================

        # Convert token IDs into embeddings
        x = self.get_embedding(input_ids)

        # Pass through Transformer backbone
        for block in self.blocks:
            x = block(x)

        # Final normalization
        h_main = self.norm_f(x)

        # Vocabulary logits
        logits_main = self.get_output_logits(
            h_main
        )

        # Store all prediction logits
        all_logits = [logits_main]

        # Previous hidden state used by MTP chain
        h_prev = h_main

        # ====================================================
        # MULTI-TOKEN PREDICTION CHAIN
        # ====================================================

        for depth_k in range(
            1,
            self.num_mtp_heads + 1
        ):

            # Remaining valid sequence length
            L = S - depth_k

            # Stop if sequence becomes empty
            if L <= 0:
                break

            # ------------------------------------
            # Slice previous hidden states
            # ------------------------------------
            h_prev_sliced = h_prev[:, :L, :]

            # ------------------------------------
            # Obtain future token IDs
            # ------------------------------------
            future_token_ids = input_ids[
                :,
                depth_k:depth_k + L
            ]

            # Convert future tokens to embeddings
            future_token_embeds = (
                self.get_embedding(
                    future_token_ids
                )
            )

            # ------------------------------------
            # Run corresponding MTP module
            # ------------------------------------
            h_curr = self.mtp_modules[
                depth_k - 1
            ](
                h_prev_sliced,
                future_token_embeds
            )

            # Convert hidden states to logits
            logits_k = self.get_output_logits(
                h_curr
            )

            # Save MTP logits
            all_logits.append(logits_k)

            # Feed current hidden state to next MTP
            h_prev = h_curr

        # ====================================================
        # LOSS COMPUTATION
        # ====================================================

        loss = None

        # Only compute loss during training
        if targets is not None:

            # ------------------------------------------------
            # MAIN NEXT-TOKEN LOSS
            # ------------------------------------------------

            # Remove last prediction
            shift_logits = logits_main[
                :,
                :-1,
                :
            ].contiguous()

            # Remove first target token
            shift_targets = targets[
                :,
                1:
            ].contiguous()

            # Standard GPT loss
            total_loss = F.cross_entropy(

                shift_logits.view(
                    -1,
                    self.vocab_size
                ),

                shift_targets.view(-1)
            )

            # ------------------------------------------------
            # MTP LOSSES
            # ------------------------------------------------

            mtp_loss_sum = 0.0

            # Loop over all MTP heads
            for k, logits_k in enumerate(
                all_logits[1:],
                start=1
            ):

                # Align predictions and targets
                mtp_logits = logits_k[
                    :,
                    :-1,
                    :
                ].contiguous()

                # Future targets
                mtp_targets = targets[
                    :,
                    k + 1:
                    k + 1 + mtp_logits.size(1)
                ].contiguous()

                # Skip empty tensors
                if mtp_targets.numel() == 0:
                    continue

                # Cross-entropy loss for head k
                loss_mtp_k = F.cross_entropy(

                    mtp_logits.view(
                        -1,
                        self.vocab_size
                    ),

                    mtp_targets.view(-1)
                )

                # Add to total MTP loss
                mtp_loss_sum += loss_mtp_k

            # ------------------------------------------------
            # FINAL DEEPSEEK LOSS
            # ------------------------------------------------
            #
            # L = L_main + (λ/D) Σ L_MTP
            #

            if (
                self.num_mtp_heads > 0
                and mtp_loss_sum > 0
            ):

                mtp_loss_weighted = (

                    self.mtp_loss_weight
                    / self.num_mtp_heads

                ) * mtp_loss_sum

                total_loss += (
                    mtp_loss_weighted
                )

            # Final training loss
            loss = total_loss

        # ====================================================
        # RETURN OUTPUTS
        # ====================================================

        return {

            # Main logits + MTP logits
            "logits_all": all_logits,

            # Training loss
            "loss": loss
        }

# Listing 5.5: Verifying the Causal MTP Implementation

In [5]:
def verify_deepseek_v3_mtp():

    # --------------------------------------------------------
    # MODEL CONFIGURATION
    # --------------------------------------------------------

    # Vocabulary size
    # Total number of unique tokens
    vocab_size = 1000

    # Embedding dimension
    # Size of each token vector
    d_model = 128

    # Number of Transformer layers
    num_layers = 6

    # Number of attention heads
    nhead = 8

    # Number of MTP heads (D)
    # Model predicts multiple future tokens
    num_mtp_heads = 3

    # Feed Forward Network size
    dim_feedforward = 512

    # λ (lambda) for MTP loss
    mtp_loss_weight = 0.1

    # --------------------------------------------------------
    # CREATE MODEL
    # --------------------------------------------------------

    model = DeepSeekV3WithMTP(

        # Vocabulary size
        vocab_size=vocab_size,

        # Hidden dimension
        d_model=d_model,

        # Number of Transformer blocks
        num_layers=num_layers,

        # Attention heads
        nhead=nhead,

        # Number of MTP modules
        num_mtp_heads=num_mtp_heads,

        # FFN dimension
        dim_feedforward=dim_feedforward,

        # MTP loss weight
        mtp_loss_weight=mtp_loss_weight
    )

    # Print total parameter count
    print(
        f"Model created with "
        f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params"
    )

    # --------------------------------------------------------
    # CREATE DUMMY TEST DATA
    # --------------------------------------------------------

    # Number of sequences in batch
    batch_size = 2

    # Tokens per sequence
    seq_len = 20

    # Generate random token IDs
    #
    # Shape:
    # (2,20)
    #
    # Values:
    # 0 to 999
    input_ids = torch.randint(
        0,
        vocab_size,
        (batch_size, seq_len)
    )

    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------

    # Run model
    #
    # targets=input_ids
    # is used only for testing loss calculation
    outputs = model(
        input_ids,
        targets=input_ids
    )

    # Extract logits
    all_logits = outputs['logits_all']

    # Extract total loss
    loss = outputs['loss']

    # --------------------------------------------------------
    # VERIFY OUTPUT SHAPES
    # --------------------------------------------------------

    print("\nLogits shapes:")

    # Loop through main output + MTP outputs
    for i, logits in enumerate(all_logits):

        # Main model output
        if i == 0:
            pred_type = "Main"

        # MTP outputs
        else:
            pred_type = f"MTP k={i}"

        # Print tensor shape
        print(
            f"  {pred_type:10}: "
            f"{list(logits.shape)}"
        )

    # --------------------------------------------------------
    # PRINT LOSS
    # --------------------------------------------------------

    print(
        f"\nTotal loss: "
        f"{loss.item():.4f}"
    )

# ============================================================
# RUN TEST
# ============================================================

verify_deepseek_v3_mtp()

Model created with 2.01M params

Logits shapes:
  Main      : [2, 20, 1000]
  MTP k=1   : [2, 19, 1000]
  MTP k=2   : [2, 18, 1000]
  MTP k=3   : [2, 17, 1000]

Total loss: 116.6747


# Part 2: FP8 Quantization

In [6]:
def basic_quantize(
    tensor: torch.Tensor,
    target_max: float = 127.0
):
    """
    Demonstrates the basic scaling mechanism for quantization.

    Converts floating-point values into INT8 values
    by mapping them into the range:

    [-127, 127]
    """

    # --------------------------------------------------------
    # STEP 1: Find Maximum Absolute Value (α)
    # --------------------------------------------------------
    #
    # Determines the largest magnitude value
    # in the tensor.
    #
    # Example:
    # [-7.59, 2.31, 10.80]
    #
    # alpha = 10.80
    #
    alpha = tensor.abs().max()

    # --------------------------------------------------------
    # STEP 2: Compute Scale Factor
    # --------------------------------------------------------
    #
    # Formula:
    #
    # scale = target_max / alpha
    #
    # Example:
    #
    # scale = 127 / 10.80
    #       = 11.759
    #
    # Small epsilon avoids division by zero.
    #
    scale = target_max / (alpha + 1e-8)

    # --------------------------------------------------------
    # STEP 3: Quantization
    # --------------------------------------------------------
    #
    # Multiply values by scale
    # Round to nearest integer
    # Clamp to INT8 range
    #
    # Example:
    #
    # 10.80 × 11.759 ≈ 127
    # 2.31 × 11.759 ≈ 27
    #
    quantized = torch.round(
        tensor * scale
    ).clamp(
        -target_max,
        target_max
    )

    # Return INT8 representation and scale
    return quantized, scale


# ============================================================
# Dequantization Function
# ============================================================

def basic_dequantize(
    quantized: torch.Tensor,
    scale: float
):
    """
    Recovers approximate floating-point values
    from quantized integers.
    """

    # --------------------------------------------------------
    # STEP 4: Dequantization
    # --------------------------------------------------------
    #
    # Formula:
    #
    # original ≈ quantized / scale
    #
    # Example:
    #
    # 127 / 11.759
    # ≈ 10.80
    #
    return quantized / scale


# ============================================================
# DEMONSTRATION
# ============================================================

# Original floating-point tensor
original = torch.tensor(
    [-7.59, 2.31, 10.80, -4.12, 0.55]
)

# Print original values
print(
    f"Original values: "
    f"{original.tolist()}"
)

# Quantize tensor
quantized, scale = basic_quantize(
    original,
    target_max=127.0
)

# Print quantized values
print(
    f"Quantized (INT8): "
    f"{quantized.tolist()}"
)

# Print scale factor
print(
    f"Scaling factor: "
    f"{scale.item():.4f}"
)

# Recover approximate values
recovered = basic_dequantize(
    quantized,
    scale
)

# Print reconstructed values
print(
    f"De-quantized values: "
    f"{recovered.tolist()}"
)

# Compute absolute quantization error
print(
    f"Quantization error: "
    f"{(original - recovered).abs().tolist()}"
)

Original values: [-7.590000152587891, 2.309999942779541, 10.800000190734863, -4.119999885559082, 0.550000011920929]
Quantized (INT8): [-89.0, 27.0, 127.0, -48.0, 6.0]
Scaling factor: 11.7593
De-quantized values: [-7.5685038566589355, 2.29606294631958, 10.800000190734863, -4.081889629364014, 0.5102362036705017]
Quantization error: [0.021496295928955078, 0.013936996459960938, 0.0, 0.03811025619506836, 0.039763808250427246]


## The Five Pillars of DeepSeek's FP8 Training

DeepSeek's approach is not a single technique but a sophisticated framework composed of **five key innovations**:

1. **The Mixed Precision Framework** — Strategic use of different numerical formats for different operations
2. **Fine-Grained Quantization** — Block-wise scaling instead of per-tensor scaling
3. **Increasing Accumulation Precision** — Higher precision for matrix multiplication accumulators
4. **Mantissa Over Exponents (E4M3)** — Choosing precision over range for training stability
5. **Online Quantization** — Computing scaling factors from the current batch, not delayed statistics

# Pillar 1: The Mixed Precision Framework

# Pillar 2: Fine-Grained Quantization

In [7]:
# This cell compares:
#
# 1. Traditional Per-Tensor Quantization
#    → One scale factor for the entire tensor
#
# 2. DeepSeek Fine-Grained Quantization
#    → Different scale factors for different tensor regions
#
# ============================================================
# Per-Tensor Quantization
# ============================================================

def per_tensor_quantize(
    tensor: torch.Tensor,
    target_max: float = 448.0
):
    """
    Standard per-tensor quantization.

    Uses ONE scale factor for the entire tensor.

    target_max=448 corresponds approximately to the
    maximum representable value in FP8 E4M3 format.
    """

    # --------------------------------------------------------
    # STEP 1: Find global maximum value
    # --------------------------------------------------------
    #
    # Entire tensor shares one alpha
    #
    alpha = tensor.abs().max()

    # --------------------------------------------------------
    # STEP 2: Compute global scale
    # --------------------------------------------------------
    #
    # One scale factor for all values
    #
    scale = target_max / (alpha + 1e-8)

    # --------------------------------------------------------
    # STEP 3: Quantize
    # --------------------------------------------------------
    #
    # Apply same scale everywhere
    #
    quantized = (
        tensor * scale
    ).clamp(
        -target_max,
        target_max
    )

    # --------------------------------------------------------
    # STEP 4: Dequantize
    # --------------------------------------------------------
    #
    dequantized = quantized / scale

    # Return reconstructed tensor
    # and scale factor
    return dequantized, scale


# ============================================================
# Fine-Grained Quantization
# ============================================================

def fine_grained_quantize(
    tensor: torch.Tensor,
    tile_size: int = 128,
    target_max: float = 448.0
):
    """
    DeepSeek-style fine-grained quantization.

    Instead of one scale factor:

        Whole Tensor → One Scale

    we use:

        Tile 1 → Scale 1
        Tile 2 → Scale 2
        Tile 3 → Scale 3
        ...

    This improves numerical accuracy.
    """

    # --------------------------------------------------------
    # Tensor dimensions
    # --------------------------------------------------------
    rows, cols = tensor.shape

    # --------------------------------------------------------
    # Padding calculation
    # --------------------------------------------------------
    #
    # Ensure column count becomes divisible
    # by tile_size.
    #
    pad_cols = (
        tile_size
        - cols % tile_size
    ) % tile_size

    # --------------------------------------------------------
    # Pad tensor if needed
    # --------------------------------------------------------
    if pad_cols > 0:

        tensor_padded = F.pad(

            tensor,

            # Pad only columns
            (0, pad_cols)
        )

    else:

        tensor_padded = tensor

    # --------------------------------------------------------
    # Create tiles
    # --------------------------------------------------------
    #
    # Example:
    #
    # [4,256]
    #
    # becomes
    #
    # [4,2,128]
    #
    num_tiles = (
        tensor_padded.shape[1]
        // tile_size
    )

    tiles = tensor_padded.view(

        rows,

        num_tiles,

        tile_size
    )

    # --------------------------------------------------------
    # Compute scale for each tile
    # --------------------------------------------------------
    #
    # Shape:
    #
    # [rows, num_tiles, 1]
    #
    alpha_per_tile = tiles.abs().amax(

        dim=-1,

        keepdim=True
    )

    scales = target_max / (
        alpha_per_tile + 1e-8
    )

    # --------------------------------------------------------
    # Quantize each tile separately
    # --------------------------------------------------------
    quantized_tiles = (

        tiles * scales

    ).clamp(

        -target_max,

        target_max
    )

    # --------------------------------------------------------
    # Dequantize
    # --------------------------------------------------------
    dequantized_tiles = (

        quantized_tiles / scales
    )

    # --------------------------------------------------------
    # Restore original shape
    # --------------------------------------------------------
    dequantized = (

        dequantized_tiles

        .view(rows, -1)

        [:, :cols]
    )

    return dequantized, scales


# ============================================================
# DEMONSTRATION
# ============================================================

# Set random seed
# Ensures reproducible results
torch.manual_seed(42)

# ------------------------------------------------------------
# Create synthetic tensor
# ------------------------------------------------------------
#
# Shape:
# [4,256]
#
tensor = torch.randn(4, 256)

# ------------------------------------------------------------
# Create large outliers
# ------------------------------------------------------------
#
# First 64 columns contain huge values
#
tensor[:, :64] *= 100

# ------------------------------------------------------------
# Create tiny values
# ------------------------------------------------------------
#
# Remaining columns contain very small values
#
tensor[:, 64:] *= 0.01

# ------------------------------------------------------------
# Per-tensor quantization
# ------------------------------------------------------------
deq_per_tensor, _ = per_tensor_quantize(
    tensor
)

# ------------------------------------------------------------
# Fine-grained quantization
# ------------------------------------------------------------
deq_fine_grained, _ = fine_grained_quantize(

    tensor,

    tile_size=128
)

# ------------------------------------------------------------
# Compute reconstruction errors
# ------------------------------------------------------------

error_per_tensor = (

    tensor
    - deq_per_tensor

).abs().mean()

error_fine_grained = (

    tensor
    - deq_fine_grained

).abs().mean()

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print(
    f"Per-tensor quantization error: "
    f"{error_per_tensor.item():.6f}"
)

print(
    f"Fine-grained quantization error: "
    f"{error_fine_grained.item():.6f}"
)

print(
    f"Error reduction: "
    f"{error_per_tensor / error_fine_grained:.2f}x"
)

Per-tensor quantization error: 0.000000
Fine-grained quantization error: 0.000000
Error reduction: 1.84x


### Pillar 3: Increasing Accumulation Precision

### Pillar 4: Mantissa Over Exponents (E4M3 vs E5M2)

### Pillar 5: Online Quantization

In [8]:
# Demonstrates one of DeepSeek-V3's important FP8 training innovations:
#
# 1. Traditional Delayed Scaling
# 2. DeepSeek Online Quantization
#
# ============================================================
# Online Quantization
# ============================================================

def online_quantize(
    tensor: torch.Tensor,
    target_max: float = 448.0
):
    """
    Online quantization.

    DeepSeek computes scaling factors using
    CURRENT tensor statistics.

    Every forward pass gets a fresh scale.
    """

    # --------------------------------------------------------
    # Find maximum absolute value
    # --------------------------------------------------------
    #
    # Uses CURRENT tensor values
    #
    alpha = tensor.abs().max()

    # --------------------------------------------------------
    # Compute scale
    # --------------------------------------------------------
    #
    # Scale = FP8_Max / Current_Max
    #
    scale = target_max / (alpha + 1e-8)

    # --------------------------------------------------------
    # Quantize tensor
    # --------------------------------------------------------
    #
    quantized = (

        tensor * scale

    ).clamp(

        -target_max,

        target_max
    )

    # Return quantized tensor
    # and fresh scale
    return quantized, scale


# ============================================================
# Delayed Scaling Quantizer
# ============================================================

class DelayedScalingQuantizer:
    """
    Traditional delayed scaling.

    Uses statistics from the PREVIOUS step.

    Problem:
    If data distribution changes quickly,
    the scale becomes outdated.
    """

    def __init__(
        self,
        target_max: float = 448.0
    ):

        # Maximum representable FP8 value
        self.target_max = target_max

        # Previous maximum value
        #
        # Initially unknown
        self.prev_alpha = None

    # ========================================================
    # Quantization Function
    # ========================================================
    def quantize(
        self,
        tensor: torch.Tensor
    ):

        # ----------------------------------------------------
        # First step
        # ----------------------------------------------------
        #
        # No previous statistics available
        #
        if self.prev_alpha is None:

            alpha = tensor.abs().max()

        # ----------------------------------------------------
        # Delayed scaling
        # ----------------------------------------------------
        #
        # Use previous step statistics
        #
        else:

            alpha = self.prev_alpha

        # ----------------------------------------------------
        # Compute scale
        # ----------------------------------------------------
        scale = self.target_max / (

            alpha + 1e-8
        )

        # ----------------------------------------------------
        # Quantize tensor
        # ----------------------------------------------------
        quantized = (

            tensor * scale

        ).clamp(

            -self.target_max,

            self.target_max
        )

        # ----------------------------------------------------
        # Save current statistics
        # ----------------------------------------------------
        #
        # Used in next iteration
        #
        self.prev_alpha = (

            tensor.abs()

            .max()

            .detach()
        )

        return quantized, scale


# ============================================================
# COMPARISON EXPERIMENT
# ============================================================

# Fix random seed
torch.manual_seed(42)

# Create delayed quantizer
delayed_quantizer = DelayedScalingQuantizer()

print(
    "Simulating 5 steps with changing "
    "data distributions:\n"
)

# ============================================================
# Simulate Multiple Training Steps
# ============================================================

for step in range(5):

    # --------------------------------------------------------
    # Simulate changing tensor magnitudes
    # --------------------------------------------------------
    #
    # Creates:
    #
    # Step 0 -> 0.01
    # Step 1 -> 0.1
    # Step 2 -> 1.0
    # Step 3 -> 10.0
    # Step 4 -> 100.0
    #
    magnitude = 10 ** (step - 2)

    # Create random tensor
    tensor = (

        torch.randn(64, 64)

        * magnitude
    )

    # ========================================================
    # ONLINE QUANTIZATION
    # ========================================================

    # Quantize using current statistics
    q_online, s_online = online_quantize(
        tensor
    )

    # Dequantize
    deq_online = q_online / s_online

    # Reconstruction error
    err_online = (

        tensor
        - deq_online

    ).abs().mean()

    # ========================================================
    # DELAYED SCALING
    # ========================================================

    # Quantize using previous statistics
    q_delayed, s_delayed = (

        delayed_quantizer.quantize(
            tensor
        )
    )

    # Dequantize
    deq_delayed = q_delayed / s_delayed

    # Reconstruction error
    err_delayed = (

        tensor
        - deq_delayed

    ).abs().mean()

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(

        f"Step {step} "

        f"(magnitude={magnitude:>6.2f}): "

        f"Online err={err_online.item():.6f}, "

        f"Delayed err={err_delayed.item():.6f}"
    )

Simulating 5 steps with changing data distributions:

Step 0 (magnitude=  0.01): Online err=0.000000, Delayed err=0.000000
Step 1 (magnitude=  0.10): Online err=0.000000, Delayed err=0.047852
Step 2 (magnitude=  1.00): Online err=0.000000, Delayed err=0.508181
Step 3 (magnitude= 10.00): Online err=0.000000, Delayed err=4.556325
Step 4 (magnitude=100.00): Online err=0.000000, Delayed err=47.418194


# Putting It All Together: FP8 Linear Layer


In [9]:
# Cell Combines:
# 1. Fine-Grained (Tile-Wise) Quantization
# 2. Online Scaling
# 3. FP32 Master Weights
# 4. Simulated FP8 Computation
#

class FP8Linear(nn.Module):
    """
    Simulated FP8 Linear Layer.

    DeepSeek-V3 stores master weights in FP32
    but performs computation using low-precision FP8 values.

    Features:
    - Fine-grained quantization
    - Online scaling
    - FP32 accumulation
    """

    # ========================================================
    # Constructor
    # ========================================================
    def __init__(
        self,
        in_features: int,
        out_features: int,
        tile_size: int = 128
    ):

        # Initialize parent module
        super().__init__()

        # ----------------------------------------------------
        # Master Weight Matrix
        # ----------------------------------------------------
        #
        # Shape:
        # [out_features, in_features]
        #
        # Stored in full precision (FP32)
        #
        self.weight = nn.Parameter(

            torch.randn(
                out_features,
                in_features
            ) * 0.02
        )

        # Bias vector
        self.bias = nn.Parameter(

            torch.zeros(out_features)
        )

        # Tile size used in fine-grained quantization
        self.tile_size = tile_size

        # Maximum representable FP8 E4M3 value
        self.fp8_max = 448.0

    # ========================================================
    # Fine-Grained Quantization
    # ========================================================
    def _quantize_tile_wise(
        self,
        tensor: torch.Tensor
    ) -> tuple:

        """
        Apply tile-wise FP8 quantization.

        Each tile receives its own scale factor.

        This simulates DeepSeek's fine-grained FP8 training.
        """

        # Save original shape
        orig_shape = tensor.shape

        # ----------------------------------------------------
        # Convert to 2D
        # ----------------------------------------------------
        #
        # Example:
        #
        # [B,S,D]
        #
        # becomes
        #
        # [B*S,D]
        #
        if tensor.dim() > 2:

            tensor_2d = tensor.reshape(

                -1,

                tensor.shape[-1]
            )

        else:

            tensor_2d = tensor

        # ----------------------------------------------------
        # Tensor dimensions
        # ----------------------------------------------------
        rows, cols = tensor_2d.shape

        # ----------------------------------------------------
        # Calculate required padding
        # ----------------------------------------------------
        pad_cols = (

            self.tile_size
            - cols % self.tile_size

        ) % self.tile_size

        # ----------------------------------------------------
        # Pad tensor if needed
        # ----------------------------------------------------
        if pad_cols > 0:

            tensor_padded = F.pad(

                tensor_2d,

                (0, pad_cols)
            )

        else:

            tensor_padded = tensor_2d

        # ----------------------------------------------------
        # Create tiles
        # ----------------------------------------------------
        #
        # Example:
        #
        # [32,256]
        #
        # becomes
        #
        # [32,2,128]
        #
        num_tiles = (

            tensor_padded.shape[1]
            // self.tile_size
        )

        tiles = tensor_padded.view(

            rows,

            num_tiles,

            self.tile_size
        )

        # ----------------------------------------------------
        # Online Scaling
        # ----------------------------------------------------
        #
        # Compute scale from CURRENT tile values
        #
        alpha = tiles.abs().amax(

            dim=-1,

            keepdim=True
        )

        scales = (

            self.fp8_max

            / (alpha + 1e-8)
        )

        # ----------------------------------------------------
        # Simulated FP8 Quantization
        # ----------------------------------------------------
        #
        # Multiply by scale
        # Round
        # Clamp
        #
        quantized = (

            torch.round(
                tiles * scales
            )

        ).clamp(

            -self.fp8_max,

            self.fp8_max
        )

        # ----------------------------------------------------
        # Simulated FP8 Dequantization
        # ----------------------------------------------------
        #
        # Recover approximate values
        #
        dequantized = (

            quantized / scales

        ).view(

            rows,

            -1

        )[:, :cols]

        # ----------------------------------------------------
        # Restore original shape
        # ----------------------------------------------------
        if len(orig_shape) > 2:

            dequantized = (

                dequantized.reshape(
                    orig_shape
                )
            )

        return dequantized

    # ========================================================
    # Forward Pass
    # ========================================================
    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        # ----------------------------------------------------
        # Quantize Activations
        # ----------------------------------------------------
        #
        # Input activations simulated as FP8
        #
        x_fp8 = self._quantize_tile_wise(x)

        # ----------------------------------------------------
        # Quantize Weights
        # ----------------------------------------------------
        #
        # Weight matrix simulated as FP8
        #
        w_fp8 = self._quantize_tile_wise(
            self.weight
        )

        # ----------------------------------------------------
        # Matrix Multiplication
        # ----------------------------------------------------
        #
        # Performs:
        #
        # Y = XW + b
        #
        # Accumulation remains FP32
        #
        output = F.linear(

            x_fp8,

            w_fp8,

            self.bias
        )

        return output


# ============================================================
# TEST FP8 LINEAR LAYER
# ============================================================

# Create simulated FP8 layer
fp8_layer = FP8Linear(

    256,      # Input dimension

    512,      # Output dimension

    tile_size=128
)

# Create standard FP32 layer
std_layer = nn.Linear(

    256,

    512
)

# ------------------------------------------------------------
# Copy identical weights
# ------------------------------------------------------------
#
# Ensures fair comparison
#
with torch.no_grad():

    std_layer.weight.copy_(
        fp8_layer.weight
    )

    std_layer.bias.copy_(
        fp8_layer.bias
    )

# ------------------------------------------------------------
# Create test input
# ------------------------------------------------------------
#
# Shape:
# [batch=2, seq=16, hidden=256]
#
test_input = torch.randn(

    2,

    16,

    256
)

# FP8 forward pass
out_fp8 = fp8_layer(test_input)

# Standard FP32 forward pass
out_std = std_layer(test_input)

# ------------------------------------------------------------
# Compare outputs
# ------------------------------------------------------------

print(
    f"FP8 output shape: "
    f"{list(out_fp8.shape)}"
)

print(
    f"Std output shape: "
    f"{list(out_std.shape)}"
)

print(
    f"Mean abs error: "
    f"{(out_fp8 - out_std).abs().mean().item():.6f}"
)

print(
    f"Relative error: "
    f"{((out_fp8 - out_std).abs() / (out_std.abs() + 1e-8)).mean().item():.4%}"
)

FP8 output shape: [2, 16, 512]
Std output shape: [2, 16, 512]
Mean abs error: 0.000671
Relative error: 1.5867%
